# Global Deforestation Analysis (1990–2025)
## Notebook 04: Direct Physical Causes, Agricultural Commodities & Multi-Country Comparison

### Scientific Framework & Methodology
Analyzing the causes of deforestation requires distinguishing between:
1. **Direct Proximate Physical Drivers (Curtis et al., Science 2018)**:
   - **Permanent Agriculture**: Conversion of forest to pastureland or croplands with persistent non-forest canopy.
   - **Logging**: Commercial forestry and rotational harvesting where tree cover regenerates.
   - **Wildfire**: Burning of forest canopy without conversion to alternative land uses.
   - **Shifting Cultivation**: Small-scale slash-and-burn agriculture with subsequent fallow recovery.
   - **Settlements and Infrastructure**: Urban sprawl, road construction, reservoirs.
   - **Mining & Energy**: Mineral extraction, oil/gas infrastructure.
2. **Inferred Agricultural Commodity Attribution (Pendrill et al. 2022)**:
   - Coupling satellite deforestation maps with FAOSTAT trade data (Cattle: Beef, Leather & Pasture; Oil Palm; Soybeans; Wood Fiber; Cocoa; Coffee).
3. **Multi-Country Comparison**:
   - Normalized Driver Profile Fingerprints (%) via radar charts.
   - Annual Deforestation Intensity Heatmaps detecting temporal loss spikes.


In [1]:
import os, sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJ_ROOT)
from src.utils import load_processed_data, THEME, DRIVER_COLORS
from src.charts import (
    create_drivers_pie_chart,
    create_drivers_stacked_bar,
    create_commodity_breakdown_chart,
    create_country_cause_radar,
    create_country_heatmap
)

data = load_processed_data()
df_drivers = data["drivers"]
df_comm = data["commodities"]
print("Drivers and Commodity datasets loaded successfully.")


Drivers and Commodity datasets loaded successfully.


### 1. Global Breakdown of Measured Direct Drivers (2001–2024)
What are the primary physical drivers of global tree cover loss over the last two decades?


In [2]:
fig_pie = create_drivers_pie_chart(df_drivers, entity="World", year_range=(2001, 2024))
fig_pie.show()


### 2. Longitudinal Dynamics of Direct Drivers (2001–2024)
Stacked temporal breakdown demonstrating annual fluctuations and the dramatic surge in wildfire-driven loss in recent years.


In [3]:
fig_stacked = create_drivers_stacked_bar(df_drivers, entity="World", start_year=2001, end_year=2024)
fig_stacked.show()


### 3. Agricultural Commodity Attribution (Pendrill et al. / Singh et al. 2026)
Which commercial agricultural commodities drive the largest shares of deforestation?
- **Cattle (Beef, Leather & Pasture)**: #1 driver globally, accounting for more tropical deforestation than the next three commodities combined.
- **Soybeans**: Concentrated in the Amazon and Cerrado biomes of Brazil and Bolivia.
- **Oil Palm**: Concentrated in Indonesia and Malaysia.
- **Forestry (Paper & Wood Fiber)**: Commercial pulp and timber plantations.
- **Cocoa & Coffee**: Tropical cash crops in West Africa and Latin America.


In [4]:
fig_comm = create_commodity_breakdown_chart(df_comm, entity="World", year_range=(2001, 2023))
fig_comm.show()


### 4. Continental Sunburst of Direct Drivers
Visualizing hierarchical distribution of drivers across continents.


In [5]:
sun_rows = []
drv_cols = [c for c in DRIVER_COLORS.keys() if c in df_drivers.columns and c != "Other natural disturbances"]
sub_sun = df_drivers[(df_drivers["Year"] >= 2001) & (df_drivers["Year"] <= 2024)]

for cont in ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]:
    c_sub = sub_sun[sub_sun["Continent"] == cont]
    for d in drv_cols:
        val = c_sub[d].sum()
        if val > 0:
            sun_rows.append({"Continent": cont, "Driver": d, "Area_Mha": val / 1e6})

df_sun = pd.DataFrame(sun_rows)
fig_sun = px.sunburst(
    df_sun,
    path=["Continent", "Driver"],
    values="Area_Mha",
    color="Driver",
    color_discrete_map=DRIVER_COLORS,
    title="Continental Share of Direct Drivers (2001–2024) [Mha]"
)
fig_sun.update_layout(template="plotly_white")
fig_sun.show()


### 5. Multi-Country Comparison: Driver Fingerprint Radar Chart
Comparing normalized driver distributions across key deforestation frontier and transition nations (Brazil, Indonesia, DRC, China, Canada).


In [6]:
compare_isos = ["BRA", "IDN", "COD", "CHN", "CAN"]
fig_radar = create_country_cause_radar(df_drivers, compare_isos, year_range=(2001, 2024))
fig_radar.show()


### 6. Multi-Country Comparison: Annual Intensity Spike Heatmap
Detecting specific loss spikes and fire seasons across the compared nations.


In [7]:
fig_heat = create_country_heatmap(
    df_drivers,
    "Total_Loss_ha",
    "All Direct Causes",
    iso3_list=compare_isos,
    year_range=(2001, 2024),
    top_n=5
)
fig_heat.show()


### Statistical Findings & Etiological Conclusions
1. **Permanent Agriculture is the Primary Tropical Threat**: Permanent agricultural conversion accounts for the vast majority of irreversible tropical deforestation, whereas forestry in temperate/boreal zones largely permits canopy regeneration.
2. **Cattle Ranching Dominance**: Beef production drives the largest share of commodity deforestation worldwide, requiring extensive pastures in Latin America.
3. **The Wildfire Escalation**: Wildfires have exhibited unprecedented spikes in Canada, Russia, and the western United States, reflecting positive climate feedback loops between atmospheric warming, drought, and forest flammability.
